# Final Realtime Variant Benchmark

Ce notebook reprend le script `final_realtime_variant_benchmark.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Benchmark temps reel des variantes causales de deploiement.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Benchmark causal final-risk runtime variants on a raw video.
- Commande de reproduction referencee : optimized real-time variant benchmark, CPU-side lightweight runtime benchmark, ONNX pose backend runtime smoke.
- Artefacts controles : Optimized causal runtime variant benchmark exists. (`runs/exp_053_realtime_variant_benchmark/runtime_variant_total_summary.csv`); CPU-side lightweight runtime benchmark exists. (`runs/exp_055_realtime_variant_benchmark_cpu_light/runtime_variant_total_summary.csv`).
- Le script ecrit ou lit des artefacts experimentaux dans `runs/`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "final_realtime_variant_benchmark.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import time
from collections import deque
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image

from final_fused_inference_demo import (
    BASE_MODELS,
    FINAL_SCORE_COLS,
    apply_final_formulas,
    crop_transform,
    load_crop_checkpoint,
    load_sequence_checkpoint,
    resolve,
    safe_person_crop,
    selected_crop_arches,
    write_contact_sheet,
)
from ml_pipeline import TRACKED_PARTS, YOLO_POSE_WEIGHTS, load_dataset, load_json, pose_rows_for_results, write_json, zone_polygon


## Fonction `choose_device`

Cette cellule definit `choose_device`. Elle prepare une partie du script.

In [ ]:
def choose_device(requested):
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(requested)


## Fonction `finite_diff`

Cette cellule definit `finite_diff`. Elle prepare une partie du script.

In [ ]:
def finite_diff(cur, prev, scale):
    try:
        cur = float(cur)
        prev = float(prev)
    except (TypeError, ValueError):
        return 0.0
    if not np.isfinite(cur) or not np.isfinite(prev):
        return 0.0
    return float((cur - prev) * scale)


## Classe `IncrementalFeatureBuilder`

Cette cellule definit `IncrementalFeatureBuilder`. Elle prepare une partie du script.

In [ ]:
class IncrementalFeatureBuilder:
    def __init__(self, feature_cols):
        self.feature_cols = list(feature_cols)
        self.prev_raw = None
        self.prev_enhanced = None

    def update(self, raw_row):
        row = dict(raw_row)
        fps = float(row.get("fps", 30.0) or 30.0)
        if self.prev_raw is None:
            row["max_signed_dist_vel"] = 0.0
        else:
            row["max_signed_dist_vel"] = finite_diff(row.get("max_signed_dist_norm"), self.prev_raw.get("max_signed_dist_norm"), fps)
        if self.prev_enhanced is None:
            row["max_signed_dist_acc"] = 0.0
        else:
            row["max_signed_dist_acc"] = finite_diff(row.get("max_signed_dist_vel"), self.prev_enhanced.get("max_signed_dist_vel"), fps)

        for part in TRACKED_PARTS:
            for axis in ["x", "y"]:
                source = f"{part}_{axis}_norm"
                out = f"{part}_{axis}_vel"
                row[out] = 0.0 if self.prev_raw is None else finite_diff(row.get(source), self.prev_raw.get(source), fps)
            source = f"{part}_signed_dist_norm"
            out = f"{part}_signed_dist_vel"
            row[out] = 0.0 if self.prev_raw is None else finite_diff(row.get(source), self.prev_raw.get(source), fps)

        vec = []
        for col in self.feature_cols:
            value = row.get(col, 0.0)
            try:
                value = float(value)
            except (TypeError, ValueError):
                value = 0.0
            vec.append(value if np.isfinite(value) else 0.0)
        self.prev_raw = dict(raw_row)
        self.prev_enhanced = row
        return row, np.asarray(vec, dtype=np.float32)


## Fonction `update_ema`

Cette cellule definit `update_ema`. Elle prepare une partie du script.

In [ ]:
def update_ema(previous, value, alpha):
    if previous is None:
        return float(value)
    return float(alpha * value + (1.0 - alpha) * previous)


## Fonction `make_sequence_from_buffer`

Cette cellule definit `make_sequence_from_buffer`. Elle prepare une partie du script.

In [ ]:
def make_sequence_from_buffer(buffer, seq_len):
    values = list(buffer)
    if not values:
        raise ValueError("empty sequence buffer")
    seq = np.asarray(values[-seq_len:], dtype=np.float32)
    if len(seq) < seq_len:
        pad = np.repeat(seq[:1], seq_len - len(seq), axis=0)
        seq = np.vstack([pad, seq])
    return seq.reshape(1, seq_len, seq.shape[1]).astype(np.float32)


## Fonction `predict_crop_frame`

Cette cellule definit `predict_crop_frame`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict_crop_frame(model, frame, pose_row, transform, device):
    crop = safe_person_crop(frame, pose_row)
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    tensor = transform(Image.fromarray(rgb)).unsqueeze(0).to(device)
    return float(torch.sigmoid(model(tensor).view(-1))[0].detach().cpu())


## Fonction `variant_specs`

Cette cellule definit `variant_specs`. Elle prepare une partie du script.

In [ ]:
def variant_specs():
    return [
        {
            "variant": "research_full_3seed_learned",
            "description": "Exact 3-seed final learned-meta research stack: 9 TCNs plus selected crop models.",
            "seeds": [111, 222, 333],
            "bases": list(BASE_MODELS),
            "crop_policy": "selected",
            "crop_stride": 1,
            "score_col": "final_learned_meta_mean",
            "threshold": 0.48,
            "use_meta": True,
        },
        {
            "variant": "single_seed111_learned",
            "description": "Lower-cost learned-meta stack using seed 111 only: 3 TCNs plus selected attention/PPE crop models.",
            "seeds": [111],
            "bases": list(BASE_MODELS),
            "crop_policy": "selected",
            "crop_stride": 1,
            "score_col": "final_learned_meta_mean",
            "threshold": 0.48,
            "use_meta": True,
        },
        {
            "variant": "fast_rule_seed111_small_mobilenet",
            "description": "Fast transparent rule stack: seed 111 TCN focal plus small attention CNN and MobileNet blouse/PPE.",
            "seeds": [111],
            "bases": ["tcn_aug_focal"],
            "crop_policy": "fast_small_mobilenet",
            "crop_stride": 1,
            "score_col": "final_attention_ppe_prior",
            "threshold": 0.57,
            "use_meta": False,
        },
        {
            "variant": "fast_rule_seed111_small_mobilenet_crop_stride30",
            "description": "Fast transparent rule stack with slow context updates: seed 111 TCN focal, small attention CNN, MobileNet blouse/PPE, crop models refreshed every 30 frames.",
            "seeds": [111],
            "bases": ["tcn_aug_focal"],
            "crop_policy": "fast_small_mobilenet",
            "crop_stride": 30,
            "score_col": "final_attention_ppe_prior",
            "threshold": 0.57,
            "use_meta": False,
        },
        {
            "variant": "sequence_only_seed111_tcn_focal",
            "description": "Core real-time danger stack: seed 111 TCN focal only, no crop risk, no meta fusion.",
            "seeds": [111],
            "bases": ["tcn_aug_focal"],
            "crop_policy": "none",
            "crop_stride": 0,
            "score_col": "final_sequence_only",
            "threshold": 0.35,
            "use_meta": False,
        },
    ]


## Fonction `crop_arches_for_policy`

Cette cellule definit `crop_arches_for_policy`. Elle prepare une partie du script.

In [ ]:
def crop_arches_for_policy(fusion_run, seed, policy):
    if policy == "none":
        return None, None
    if policy == "selected":
        return selected_crop_arches(fusion_run, seed)
    if policy == "fast_small_mobilenet":
        return "small_cnn", "mobilenet_v3_small"
    raise ValueError(policy)


## Fonction `load_variant_runtime`

Cette cellule definit `load_variant_runtime`. Elle prepare une partie du script.

In [ ]:
def load_variant_runtime(spec, fusion_run, feature_cols, device, image_size):
    transform = crop_transform(image_size)
    seeds = []
    for seed in spec["seeds"]:
        normalizer = np.load(fusion_run / "features" / f"sequence_normalizer_seed{seed}.npz")
        seed_runtime = {
            "seed": int(seed),
            "mean": normalizer["mean"].astype(np.float32),
            "std": normalizer["std"].astype(np.float32),
            "buffer": deque(maxlen=120),
            "attention_ema": 0.0 if spec["crop_policy"] == "none" else None,
            "ppe_ema": 0.0 if spec["crop_policy"] == "none" else None,
            "sequence": {},
        }
        attention_arch, blouse_arch = crop_arches_for_policy(fusion_run, seed, spec["crop_policy"])
        seed_runtime["attention_arch"] = attention_arch or "none"
        seed_runtime["blouse_arch"] = blouse_arch or "none"
        if attention_arch:
            seed_runtime["attention_model"], _ = load_crop_checkpoint(fusion_run / "models" / f"attention_seed{seed}_{attention_arch}.pt", device)
            seed_runtime["blouse_model"], _ = load_crop_checkpoint(fusion_run / "models" / f"blouse_seed{seed}_{blouse_arch}.pt", device)
        for base in spec["bases"]:
            model, payload = load_sequence_checkpoint(fusion_run / "models" / f"seed{seed}_{base}.pt", device)
            item = {"model": model, "payload": payload}
            if spec["use_meta"]:
                item["meta"] = joblib.load(fusion_run / "models" / f"meta_seed{seed}_{base}.joblib")
            seed_runtime["sequence"][base] = item
        seeds.append(seed_runtime)
    return {"spec": spec, "transform": transform, "seeds": seeds, "feature_cols": feature_cols}


## Fonction `score_variant_frame`

Cette cellule definit `score_variant_frame`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def score_variant_frame(runtime, frame, pose_row, feature_vec, device, alpha, frame_pos):
    spec = runtime["spec"]
    crop_start = time.perf_counter()
    for seed_runtime in runtime["seeds"]:
        norm_vec = ((feature_vec - seed_runtime["mean"]) / seed_runtime["std"]).astype(np.float32)
        seed_runtime["buffer"].append(norm_vec)
        crop_due = spec["crop_policy"] != "none" and (seed_runtime["attention_ema"] is None or int(spec.get("crop_stride", 1)) <= 1 or frame_pos % int(spec.get("crop_stride", 1)) == 0)
        if crop_due:
            attention_now = predict_crop_frame(seed_runtime["attention_model"], frame, pose_row, runtime["transform"], device)
            ppe_now = predict_crop_frame(seed_runtime["blouse_model"], frame, pose_row, runtime["transform"], device)
            seed_runtime["attention_ema"] = update_ema(seed_runtime["attention_ema"], attention_now, alpha)
            seed_runtime["ppe_ema"] = update_ema(seed_runtime["ppe_ema"], ppe_now, alpha)
    crop_ms = (time.perf_counter() - crop_start) * 1000.0

    sequence_start = time.perf_counter()
    seed_scores = []
    for seed_runtime in runtime["seeds"]:
        seed_row = {
            "repeat_seed": seed_runtime["seed"],
            "attention_risk": float(seed_runtime["attention_ema"] or 0.0),
            "ppe_risk": float(seed_runtime["ppe_ema"] or 0.0),
        }
        for base, item in seed_runtime["sequence"].items():
            payload = item["payload"]
            X = make_sequence_from_buffer(seed_runtime["buffer"], int(payload["seq_len"]))
            xb = torch.from_numpy(X).to(device)
            probs = torch.sigmoid(item["model"](xb)).detach().cpu().numpy()[0]
            horizons = [float(h) for h in payload["horizons"]]
            h_idx = horizons.index(1.0) if 1.0 in horizons else int(np.argmin(np.abs(np.asarray(horizons) - 1.0)))
            danger = float(probs[h_idx])
            seed_row[f"sequence_{base}"] = danger
            if spec["use_meta"]:
                meta_payload = item["meta"]
                meta_input = pd.DataFrame(
                    {
                        "danger_risk_1.0s": [danger],
                        "attention_risk": [seed_row["attention_risk"]],
                        "ppe_risk": [seed_row["ppe_risk"]],
                    }
                )
                seed_row[f"learned_meta_{base}"] = float(meta_payload["model"].predict_proba(meta_input[meta_payload["meta_cols"]])[:, 1][0])
        seed_scores.append(apply_final_formulas(pd.DataFrame([seed_row])).iloc[0].to_dict())
    sequence_ms = (time.perf_counter() - sequence_start) * 1000.0
    return seed_scores, crop_ms, sequence_ms


## Fonction `extract_pose_once`

Cette cellule definit `extract_pose_once`. Elle prepare une partie du script.

In [ ]:
def extract_pose_once(raw_video, polygon, imgsz, pose_conf, pose_weights, pose_device=None, max_frames=None):
    from ultralytics import YOLO

    pose_model = YOLO(str(pose_weights))
    cap = cv2.VideoCapture(str(raw_video))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    meta = {
        "video_id": raw_video.stem,
        "path": str(raw_video),
        "fps": fps,
        "width": width,
        "height": height,
    }
    frames = []
    pose_rows = []
    pose_latencies = []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if max_frames is not None and frame_idx >= max_frames:
            break
        start = time.perf_counter()
        predict_kwargs = {"imgsz": imgsz, "conf": pose_conf, "verbose": False}
        if pose_device:
            predict_kwargs["device"] = pose_device
        result = pose_model.predict([frame], **predict_kwargs)[0]
        pose_latencies.append((time.perf_counter() - start) * 1000.0)
        pose_rows.append(pose_rows_for_results(meta, "inference", [frame_idx], [result], polygon)[0])
        frames.append(frame)
        frame_idx += 1
    cap.release()
    return frames, pose_rows, np.asarray(pose_latencies, dtype=np.float64)


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(values, warmup):
    values = np.asarray(values, dtype=np.float64)
    steady = values[warmup:] if len(values) > warmup else values
    return {
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "max_ms": float(np.max(values)),
        "steady_mean_ms": float(np.mean(steady)),
        "steady_p95_ms": float(np.percentile(steady, 95)),
        "estimated_fps_from_mean": float(1000.0 / max(1e-9, np.mean(values))),
        "estimated_fps_from_steady_mean": float(1000.0 / max(1e-9, np.mean(steady))),
    }


## Fonction `write_reference_comparison`

Cette cellule definit `write_reference_comparison`. Elle prepare une partie du script.

In [ ]:
def write_reference_comparison(out_dir, predictions, reference_path):
    reference_path = resolve(reference_path)
    if not reference_path.exists():
        return None
    ref = pd.read_csv(reference_path)
    candidate = predictions[predictions["variant"].eq("research_full_3seed_learned")].copy()
    if ref.empty or candidate.empty:
        return None
    cols = ["final_learned_meta_mean_mean3", "attention_risk_mean3", "ppe_risk_mean3"]
    available = [col for col in cols if col in ref.columns and col in candidate.columns]
    if not available:
        return None
    merged = ref[["frame", *available]].merge(candidate[["frame", *available]], on="frame", suffixes=("_reference", "_optimized"))
    rows = []
    for col in available:
        diff = (merged[f"{col}_reference"] - merged[f"{col}_optimized"]).abs()
        rows.append(
            {
                "column": col,
                "rows_compared": int(len(merged)),
                "max_abs_diff": float(diff.max()),
                "mean_abs_diff": float(diff.mean()),
                "p95_abs_diff": float(np.percentile(diff, 95)),
            }
        )
    out_path = out_dir / "optimized_reference_comparison.csv"
    pd.DataFrame(rows).to_csv(out_path, index=False)
    return out_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    out_dir = resolve(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    raw_video = resolve(args.raw_video)
    device = choose_device(args.device)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    videos, _, _, zones = load_dataset()
    polygon = zone_polygon(zones)
    fusion_run = resolve(args.fusion_run)
    feature_run = resolve(args.sequence_feature_run)
    feature_payload = load_json(feature_run / "features" / "sequence_feature_columns.json")
    feature_cols = feature_payload["feature_columns"]

    pose_weights = resolve(args.pose_weights)
    frames, pose_rows, pose_ms = extract_pose_once(
        raw_video,
        polygon,
        args.imgsz,
        args.pose_conf,
        pose_weights,
        args.pose_device,
        args.max_frames,
    )
    if not frames:
        raise SystemExit("No frames processed")

    all_prediction_rows = []
    latency_rows = []
    contact_root = out_dir / "contact_sheets"
    contact_root.mkdir(parents=True, exist_ok=True)
    specs = variant_specs()
    if args.variants:
        requested = set(args.variants)
        specs = [spec for spec in specs if spec["variant"] in requested]
        missing = sorted(requested - {spec["variant"] for spec in specs})
        if missing:
            raise SystemExit(f"Unknown variants requested: {missing}")
    for spec in specs:
        runtime = load_variant_runtime(spec, fusion_run, feature_cols, device, args.image_size)
        feature_builder = IncrementalFeatureBuilder(feature_cols)
        alarm_run = 0
        variant_rows = []
        variant_processing_ms = []
        crop_values = []
        sequence_values = []
        for idx, (frame, pose_row) in enumerate(zip(frames, pose_rows)):
            enhanced_row, feature_vec = feature_builder.update(pose_row)
            start = time.perf_counter()
            seed_scores, crop_latency, sequence_latency = score_variant_frame(runtime, frame, enhanced_row, feature_vec, device, args.crop_ema_alpha, idx)
            processing_ms = (time.perf_counter() - start) * 1000.0
            crop_values.append(crop_latency)
            sequence_values.append(sequence_latency)
            variant_processing_ms.append(processing_ms)
            row = {
                "variant": spec["variant"],
                "video_id": raw_video.stem,
                "path": str(raw_video),
                "frame": int(pose_row["frame"]),
                "time_s": float(pose_row["time_s"]),
                "fps": float(pose_row["fps"]),
                "pose_latency_ms": float(pose_ms[idx]),
                "crop_latency_ms": float(crop_latency),
                "sequence_latency_ms": float(sequence_latency),
                "variant_processing_latency_ms": float(processing_ms),
                "estimated_total_latency_ms": float(pose_ms[idx] + processing_ms),
            }
            for seed_score in seed_scores:
                seed = int(seed_score["repeat_seed"])
                for col in ["attention_risk", "ppe_risk", *FINAL_SCORE_COLS]:
                    if col in seed_score:
                        row[f"{col}_seed{seed}"] = float(seed_score[col])
            for col in ["attention_risk", "ppe_risk", *FINAL_SCORE_COLS]:
                seed_cols = [f"{col}_seed{seed}" for seed in spec["seeds"] if f"{col}_seed{seed}" in row]
                if seed_cols:
                    row[f"{col}_mean{len(seed_cols)}"] = float(np.mean([row[c] for c in seed_cols]))
            score_col = spec["score_col"]
            if score_col not in row:
                candidate = f"{score_col}_mean{len(spec['seeds'])}"
                if candidate in row:
                    score_col = candidate
                else:
                    raise SystemExit(f"Score column not found for {spec['variant']}: {spec['score_col']}")
            score = float(row[score_col])
            alarm_run = alarm_run + 1 if score >= spec["threshold"] else 0
            row["score_col"] = score_col
            row["threshold"] = float(spec["threshold"])
            row["alarm"] = int(alarm_run >= args.persistence_frames)
            variant_rows.append(row)
            all_prediction_rows.append(row)

        pred = pd.DataFrame(variant_rows)
        variant_dir = contact_root / spec["variant"]
        variant_dir.mkdir(parents=True, exist_ok=True)
        contact_sheet = write_contact_sheet(variant_dir, raw_video, pred, polygon, str(pred["score_col"].iloc[0]), float(spec["threshold"]))
        for component, values in [
            ("pose", pose_ms),
            ("crop", crop_values),
            ("sequence", sequence_values),
            ("variant_processing", variant_processing_ms),
            ("estimated_total", pred["estimated_total_latency_ms"].to_numpy(dtype=np.float64)),
        ]:
            row = {
                "variant": spec["variant"],
                "description": spec["description"],
                "component": component,
                "frames": int(len(values)),
                "warmup_excluded_frames": int(args.warmup_excluded_frames),
            }
            row.update(summarize(values, args.warmup_excluded_frames))
            latency_rows.append(row)
        config_score_col = str(pred["score_col"].iloc[0])
        write_json(
            variant_dir / "variant_config.json",
            {
                **spec,
                "rows": int(len(pred)),
                "score_col_resolved": config_score_col,
                "max_score": float(pred[config_score_col].max()),
                "first_alarm_time_s": None if pred[pred["alarm"] == 1].empty else float(pred[pred["alarm"] == 1]["time_s"].iloc[0]),
                "contact_sheet": None if contact_sheet is None else str(contact_sheet),
            },
        )

    prediction_df = pd.DataFrame(all_prediction_rows)
    latency_df = pd.DataFrame(latency_rows)
    prediction_df.to_csv(out_dir / "runtime_variant_predictions.csv", index=False)
    latency_df.to_csv(out_dir / "runtime_variant_latency_summary.csv", index=False)
    summary = (
        latency_df[latency_df["component"].eq("estimated_total")]
        .sort_values("steady_mean_ms")
        .reset_index(drop=True)
    )
    summary.to_csv(out_dir / "runtime_variant_total_summary.csv", index=False)
    reference_comparison = write_reference_comparison(out_dir, prediction_df, args.reference_causal_predictions)
    write_json(
        out_dir / "runtime_variant_benchmark_config.json",
        {
            "raw_video": str(raw_video),
            "fusion_run": str(fusion_run),
            "sequence_feature_run": str(feature_run),
            "device": str(device),
            "pose_weights": str(pose_weights),
            "pose_device": args.pose_device,
            "rows": int(len(frames)),
            "pose_policy": "YOLO pose is measured once frame-by-frame; total variant latency is pose latency plus causal variant processing latency on the same frame.",
            "feature_policy": "Incremental motion features match the sequence training feature definitions and use frames <= t only.",
            "warmup_excluded_frames": int(args.warmup_excluded_frames),
            "variants": specs,
            "reference_causal_predictions": str(resolve(args.reference_causal_predictions)),
            "reference_comparison": None if reference_comparison is None else str(reference_comparison),
        },
    )
    print(out_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Benchmark causal final-risk runtime variants on a raw video.")
    parser.add_argument("--fusion-run", default="runs/exp_020_same_split_fusion")
    parser.add_argument("--sequence-feature-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--raw-video", required=True)
    parser.add_argument("--out-dir", required=True)
    parser.add_argument("--crop-ema-alpha", type=float, default=0.20)
    parser.add_argument("--persistence-frames", type=int, default=2)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--pose-conf", type=float, default=0.10)
    parser.add_argument("--pose-weights", default=str(YOLO_POSE_WEIGHTS))
    parser.add_argument("--pose-device", default=None)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--max-frames", type=int, default=None)
    parser.add_argument("--warmup-excluded-frames", type=int, default=5)
    parser.add_argument("--reference-causal-predictions", default="runs/exp_051_causal_final_fused_stream_smoke/causal_fused_stream_predictions.csv")
    parser.add_argument("--variants", nargs="*", default=None)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement concret du benchmark temps reel
# Cette configuration reprend la video utilisee dans le run de reference.
from datetime import datetime
import sys

OUT_DIR = f"runs/exp_053_realtime_variant_benchmark_notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RAW_VIDEO = r"C:\Users\ilyas\Desktop\pose recognision\captures\ilyas\unsafe\unsafe blooza good  20260512_190912.mp4"
NOTEBOOK_ARGS = [
    "--raw-video", RAW_VIDEO,
    "--out-dir", OUT_DIR,
    "--warmup-excluded-frames", "5",
]

ancien_argv = sys.argv[:]
sys.argv = ["final_realtime_variant_benchmark.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
